# LLM Wiki：AI Agents 入門（Colab 執行版）

**先討論、後生成：** 本專案先定義讀者、範圍、核心問題與來源，再依來源建立 Wiki，最後從 Wiki 產生教學 HTML。

## 設計討論紀錄
- **目標讀者：** 沒有 AI Agent 背景的大學生。
- **核心問題：** Agent 是什麼？和聊天機器人有何不同？如何運作？Workflow／Graph 怎麼設計？有哪些風險？
- **內容邊界：** 介紹通用概念，不深入特定框架程式實作。
- **資料策略：** 優先使用 Anthropic、OpenAI、Google Cloud、IBM、LangChain 的官方文章／文件；保留來源 URL 與摘要。
- **輸出規格：** 5 篇以上來源、5 個 Wiki 主題頁、8 頁可操作 HTML 簡報。


In [ ]:
# 0. 環境設定：先把下面網址改成你自己的 GitHub repository，再執行
from pathlib import Path\nimport subprocess, os
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # ← 改成自己的 repo URL
REPO_DIR = Path('/content/llm-wiki-ai-agents')
if "YOUR_USERNAME" not in REPO_URL and not REPO_DIR.exists():
    subprocess.run(['git','clone',REPO_URL,str(REPO_DIR)], check=True)
ROOT = REPO_DIR
ROOT.mkdir(parents=True, exist_ok=True)
for d in ['sources','wiki','output','logs']:
    (ROOT/d).mkdir(exist_ok=True)
print('工作目錄：', ROOT)


In [ ]:
# 1. 定義來源（至少 5 篇）
SOURCES = [
 {'id':'S1','title':'Anthropic — Building effective agents','url':'https://www.anthropic.com/engineering/building-effective-agents',
  'summary':'以簡單、可組合的工作流程優先；區分 workflow 與 agent，並介紹 chaining、routing、parallelization、orchestrator-workers、evaluator-optimizer。'},
 {'id':'S2','title':'OpenAI — A practical guide to building agents','url':'https://cdn.openai.com/business-guides-and-resources/a-practical-guide-to-building-agents.pdf',
  'summary':'Agent 以模型管理工作流程並使用工具；核心包含模型、工具、指令與護欄，應從明確任務與可觀測性開始。'},
 {'id':'S3','title':'Google Cloud — What are AI agents?','url':'https://cloud.google.com/discover/what-are-ai-agents',
  'summary':'Agent 能感知資訊、推理、規劃並採取行動；系統可結合模型、工具、記憶與協作，需處理安全與人類監督。'},
 {'id':'S4','title':'IBM — What are AI agents?','url':'https://www.ibm.com/think/topics/ai-agents',
  'summary':'Agent 是可自主執行任務的軟體系統，具有感知、推理與行動能力；不同 Agent 的自主程度不同。'},
 {'id':'S5','title':'LangChain — LangGraph overview','url':'https://docs.langchain.com/oss/python/langgraph/overview',
  'summary':'LangGraph 用於建構具狀態、長時間執行與 human-in-the-loop 的 Agent workflow；Graph 由節點、邊與狀態構成。'}
]
assert len(SOURCES) >= 5
print('來源數量檢查：PASS', len(SOURCES))


In [ ]:
# 2. Ingest：嘗試擷取網頁正文；若網站擋爬蟲，保留人工整理的來源摘要並清楚記錄 fallback
def fetch_text(url):
    headers = {'User-Agent':'Mozilla/5.0 (compatible; LLMWikiStudentProject/1.0)'}
    r = requests.get(url, headers=headers, timeout=25)
    r.raise_for_status()
    if 'pdf' in r.headers.get('content-type','').lower() or url.lower().endswith('.pdf'):
        return f'[PDF 已成功取得，HTTP {r.status_code}，此 notebook 不做 OCR；使用來源摘要作為文字知識輸入。]'
    soup = BeautifulSoup(r.text, 'html.parser')
    for tag in soup(['script','style','nav','footer','header']):
        tag.decompose()
    text = re.sub(r'\\s+', ' ', soup.get_text(' ', strip=True))
    return text[:18000]

ingest_log = []
for item in SOURCES:
    status, content = 'fallback_summary', item['summary']
    try:
        fetched = fetch_text(item['url'])
        if len(fetched) > 300:
            status, content = 'fetched', fetched
        elif fetched.startswith('[PDF 已成功取得'):
            status = 'pdf_retrieved_summary_used'
    except Exception as e:
        status = 'fallback_summary'
        item['error'] = str(e)[:250]
    item['ingested_text'] = content
    (ROOT/'sources'/f"{item['id']}.md").write_text(
        f"# {item['title']}\\n\\nURL: {item['url']}\\n\\n擷取狀態：{status}\\n\\n## 文字／摘要\\n{content}\\n",
        encoding='utf-8')
    ingest_log.append({'id':item['id'],'title':item['title'],'status':status,'chars':len(content),'url':item['url']})
print(json.dumps(ingest_log, ensure_ascii=False, indent=2))


In [ ]:
# 3. Wiki 建置：以來源資料整理成主題頁（每個主張都保留來源對應）
pages = {
'00_首頁.md': '''# LLM Wiki：AI Agents 入門

## 主題範圍
說明 AI Agent 的定義、組件、運作循環、workflow／graph 設計、應用與風險。

## 核心摘要
AI Agent 不只是回答問題的聊天機器人，而是能在目標與限制下，運用模型判斷下一步、呼叫工具、觀察結果並持續執行的系統。

## 知識地圖
- 01 定義與差異
- 02 核心組件與循環
- 03 Workflow、Agent 與 Graph
- 04 應用、風險與治理
- 05 詞彙表

## 資料來源
S1–S5；來源全文／摘要與擷取狀態見 sources/。
''',
'01_定義與差異.md': '''# 01｜AI Agent 的定義與差異

AI Agent 是一種以目標為導向，能利用模型判斷、選擇工具、執行動作並根據結果調整下一步的系統。

## 與聊天機器人的差別
一般聊天機器人主要產生文字回答；Agent 可能拆解任務、呼叫搜尋／程式／資料庫工具，再讀取結果繼續行動。兩者不是絕對二分，聊天產品也能加入 Agent 能力。

## 判斷特徵
1. 有明確目標；2. 能決定下一步；3. 能使用工具；4. 能讀取結果並修正；5. 有停止或交接機制。

例：查三家航空行李規定，Agent 可搜尋官方資料、擷取重量與尺寸、比較差異，並標示無法確認的項目。

來源對應：S2、S3、S4。
''',
'02_核心組件與循環.md': '''# 02｜核心組件與運作循環

## 四個基本組件
- 模型：理解任務、推理、規劃與生成。
- 工具：搜尋、計算、讀寫檔案或呼叫 API。
- 指令與護欄：目標、權限、格式、禁止事項與停止條件。
- 狀態／上下文：任務進度、工具結果與必要記憶。

## 運作循環
接收目標 → 規劃 → 行動／呼叫工具 → 觀察 → 評估 → 完成或修正後重試。

循環不等於無限重試；必須設定最大迭代次數、錯誤處理、權限限制與人工確認點。

來源對應：S1、S2、S3、S4。
''',
'03_Workflow與Graph.md': '''# 03｜Workflow、Agent 與 Graph

Workflow 是開發者預先安排的步驟與路徑，較可預測；Agent 可依狀態動態決定下一步，彈性較高；混合式設計常用固定流程確保可靠性，局部交由 Agent 判斷。

## 常見模式
Prompt chaining、Routing、Parallelization、Orchestrator-workers、Evaluator-optimizer。

## Graph 三元素
Node（節點）執行工作；Edge（邊）決定流向與條件分支；State（狀態）保存資料與進度。

## 本作業 Graph
資料取得 → Wiki 整理 → 內容驗證 → HTML 生成 → 瀏覽測試。驗證失敗回到整理／取材；測試失敗回到生成。最多修正兩輪。

來源對應：S1、S5。
''',
'04_應用與風險.md': '''# 04｜應用、風險與治理

應用：研究助理、客服查詢、程式除錯、個人行政。

風險：幻覺、工具誤用、權限過大、提示注入／資料外洩、循環失控。

治理：要求來源並核對；採最小權限；付款、刪除、發送等高影響操作加入人工確認；設定最大步數、逾時與錯誤處理；記錄節點輸入輸出。

來源對應：S1–S5。
''',
'05_詞彙表.md': '''# 05｜詞彙表

| 名詞 | 意思 |
|---|---|
| Agent | 以目標為導向、能判斷並使用工具的系統 |
| Tool calling | 模型選擇並呼叫外部工具 |
| Workflow | 預先設計的步驟與路徑 |
| Graph | 以節點、邊與狀態描述流程 |
| State | 流程傳遞的資料與進度 |
| Human-in-the-loop | 由人類檢查、批准或接手 |
| Guardrails | 限制 Agent 行為的規則 |
'''
}
for name, body in pages.items():
    (ROOT/'wiki'/name).write_text(body, encoding='utf-8')
print('Wiki 頁面數量：', len(pages))


In [ ]:
# 4. 品質檢查節點：來源數、頁面數、來源對應、關鍵概念
checks = {
 '至少五份來源': len(SOURCES) >= 5,
 '至少五個 Wiki 頁面': len(pages) >= 5,
 'Wiki 有來源對應': all(('來源對應' in v or '資料來源' in v) for k,v in pages.items() if k not in ['00_首頁.md','05_詞彙表.md']),
 '包含 Agent 與 Workflow 概念': ('Agent' in pages['01_定義與差異.md'] and 'Workflow' in pages['03_Workflow與Graph.md']),
 '包含風險與治理': ('風險' in pages['04_應用與風險.md'] and '人工確認' in pages['04_應用與風險.md'])
}
for k,v in checks.items(): print(('PASS' if v else 'FAIL'), k)
if not all(checks.values()):
    raise ValueError('Wiki 品質檢查未通過：請回到 Wiki 建置節點修正後重跑。')


In [ ]:
# 5. 將 HTML 簡報寫入 output（使用 Wiki 的核心概念與來源索引）
# 若已將本專案上傳 GitHub，可直接從 repo 複製現成 HTML；這份 notebook 也會保留 HTML 產生位置。
import urllib.request
HTML_TEMPLATE_URL = None  # 可改成 GitHub raw URL；預設沿用 repo 內的 output HTML
repo_html = Path('/content/llm-wiki-ai-agents/output/AI_Agents_教學簡報.html')
if repo_html.exists():
    shutil.copy(repo_html, ROOT/'output'/'AI_Agents_教學簡報.html')
else:
    print('提醒：請先把 repo 的 output/AI_Agents_教學簡報.html 放進工作目錄；或使用本專案附帶的 HTML 檔。')


In [ ]:
# 5. HTML 簡報輸出
# 從 GitHub repo 內的 output/ 複製 HTML 到本次 Colab 工作目錄
repo_html = ROOT/'output'/'AI_Agents_教學簡報.html'
root_html = ROOT/'AI_Agents_教學簡報.html'
if root_html.exists() and not repo_html.exists():
    shutil.copy(root_html, repo_html)
if not repo_html.exists():
    raise FileNotFoundError('找不到 HTML：請確認 GitHub repo 已包含 output/AI_Agents_教學簡報.html，且已在第 0 格設定正確的 REPO_URL。')
print('HTML 已備妥：', repo_html)


## 反思（請依你實際執行結果微調）
本次最容易出錯的節點是**資料取得／擷取**：不同網站的 HTML 結構、PDF 格式與反爬蟲設定不同，可能造成正文抓取失敗或只拿到導覽文字。

因此把檢查往前移到 **Ingest 完成後、Wiki 整理前**：先記錄每個來源的擷取狀態、文字長度與 URL；若擷取失敗，就明確標記 fallback，不把摘要假裝成全文。Wiki 生成後再檢查來源對應、核心概念與風險內容。

Workflow 因此改成「取得資料 → 來源品質檢查 → Wiki 整理 → Wiki 驗證 → HTML 生成 → HTML 瀏覽測試」，並限制最多兩輪修正，避免無限循環。

> 請以你在 Colab 執行後的 `logs/execution_log.json` 為準，將實際 PASS／fallback 狀況寫進作業反思；不要宣稱未成功擷取的來源已取得全文。
